In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Load data
# I am expecting a csv file which has mainly 2 interesting columns -> Segment ids and the vehicle_roughness_observations
df = pd.read_csv("synthetic_dataset.csv")


In [ ]:
df.shape

(22, 4)

In [ ]:
df.head(5)

,observation_id,segment_id,vehicle_id,roughness_response
0,O001,A,V01,2.1
1,O002,A,V02,2.4
2,O003,A,V01,2.2
3,O004,A,V03,NaN
4,O005,A,V04,2.3


In [ ]:
print(df.isnull().sum())

observation_id        0
segment_id            1
vehicle_id            1
roughness_response    4
dtype: int64


In [ ]:

# Remove duplicate observation IDs, if present obviously
# they dont stand a chance
df = df.drop_duplicates(subset="observation_id", keep="first")


My idea is ->
1. remove the rows where segment id is missing
2. mark the rows where vehicle id is missing , and club them all as a single v_id
3. deal with the roughness_response according to the received data and its relations with other features present in the final df

In [ ]:
# Remove rows without a segment ID
df = df.dropna(subset=["segment_id"]).copy()

In [ ]:

# Convert to NaN
df["roughness_response"] = pd.to_numeric(df["roughness_response"],errors="coerce")


In [ ]:

df["roughness_was_imputed"] = df["roughness_response"].isna()

global_median = df["roughness_response"].median()
df["roughness_response"] = df.groupby("segment_id")[
    "roughness_response"
].transform(lambda x: x.fillna(x.median()))

# Fill remaining missing values using the global median
df["roughness_response"] = df["roughness_response"].fillna(global_median)


In [ ]:

# Building the stats
segment_stats = df.groupby("segment_id").agg(
    observation_count=("observation_id", "size"),
    unique_vehicle_count=("vehicle_id", "nunique"),
    roughness_count=("roughness_response", "count"),
    imputed_count=("roughness_was_imputed", "sum"),
    roughness_mean=("roughness_response", "mean"),
    roughness_variance=("roughness_response", "var")
).reset_index()


segment_stats["genuine_count"] =(segment_stats["roughness_count"]-segment_stats["imputed_count"])


In [ ]:

# for now i am calculating the confidence using genuine stuff only, excluding the imputed ones as they artificially increase the confidence

n = segment_stats["genuine_count"]
genuine_mean = (
    df.loc[~df["roughness_was_imputed"]]
    .groupby("segment_id")["roughness_response"]
    .mean()
)



In [ ]:

segment_stats["genuine_mean"] = (segment_stats["segment_id"].map(genuine_mean))
mean = segment_stats["genuine_mean"]


In [ ]:
# a hyperparameter
constt= 5
confidence_scores= n/(n+constt)

In [ ]:
segment_stats["confidence"] = confidence_scores

In [ ]:
# Save
segment_stats.to_csv("processed_road_quality.csv", index=False)

print(segment_stats)
